# Foundry Harness — Build Notebook

This is the **one notebook** the whole harness gets built in. Run cells top to bottom on a fresh Colab runtime; new roles (Indexer, Cartographer, Detector, ...) get appended as new sections at the bottom rather than as separate notebook files, so nothing later ever loses the environment an earlier section set up (installed packages, OpenAI key, in-progress SQLite database).

**Sections:**
1. **Setup** — clone, install, fetch the CodeGuard rule corpus, enter your OpenAI key. Run once per fresh runtime.
2. **Substrate** — finding store, work queue, budget governor. No LLM calls — proves the constitution's structural guarantees hold on their own, before any agent touches them.
3. *(Indexer, Cartographer, Detector, Triager, Coverage-Guide, Reporter, and the full assembled pipeline will each get their own section appended below as they're built.)*

## Setup

Clones the repo (if needed), installs dependencies, fetches the CodeGuard rule corpus, and captures your OpenAI key. Makes no LLM calls itself.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/harshamore/FoundryHarnessDC.git"
REPO_DIR = "FoundryHarnessDC"
PROJECT_SUBDIR = "langchain"  # this project lives in a subdirectory of the repo

cwd = Path.cwd()
if (cwd / "pyproject.toml").exists():
    # Already inside the project directory (e.g. running locally from repo root).
    project_path = cwd
elif (cwd.parent / "pyproject.toml").exists():
    # Running from notebooks/ inside an already-cloned checkout.
    project_path = cwd.parent
elif (cwd / REPO_DIR).exists():
    project_path = cwd / REPO_DIR / PROJECT_SUBDIR
else:
    !git clone --quiet {REPO_URL}
    project_path = cwd / REPO_DIR / PROJECT_SUBDIR

os.chdir(project_path)

# Make `foundry` importable in *this* kernel right away, without depending on
# pip's editable-install mechanism (a .pth file that Python's `site` module
# normally only processes at interpreter startup). A common Jupyter/Colab
# gotcha is `pip install -e` "succeeding" while the package still isn't
# importable until a kernel restart -- this sidesteps that entirely.
src_path = str(project_path / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Working directory: {os.getcwd()}")
print(f"'src' on sys.path: {src_path in sys.path}")

In [ ]:
%pip install --quiet -e ".[dev]"

### Fetch the CodeGuard rule corpus

Clones `cosai-oasis/project-codeguard` at a pinned commit and vendors `sources/rules/{core,owasp}` into `data/codeguard/rules/`. This has nothing to do with the Claude Code plugin on your laptop — Colab can't see that, so the harness fetches its own copy. See `docs/CODEGUARD_INTEGRATION.md`.

In [ ]:
!python scripts/fetch_codeguard_rules.py

In [ ]:
from pathlib import Path

core = list(Path("data/codeguard/rules/core").glob("*.md"))
owasp = list(Path("data/codeguard/rules/owasp").glob("*.md"))
print(f"core: {len(core)} rules, owasp: {len(owasp)} rules")
assert len(core) > 0 and len(owasp) > 0

### OpenAI API key

Entered interactively via `getpass` — never written to a file, never committed. Later sections read this from the environment. This cell itself makes no OpenAI calls.

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
print("OPENAI_API_KEY is set:", bool(os.environ.get("OPENAI_API_KEY")))

### Setup sanity check

Confirms `foundry` is importable before moving on to the Substrate section below.

In [ ]:
import importlib.util
import os

spec = importlib.util.find_spec("foundry")
if spec is None:
    raise ModuleNotFoundError(
        "'foundry' is not importable. Two likely causes, in order of likelihood:\n"
        "  1) The %pip install cell above errored or hadn't finished "
        "-- scroll up and check its output for 'Successfully installed foundry-harness'.\n"
        "  2) This cell ran in a different kernel session than the setup cell above "
        "(e.g. after Runtime > Restart session) -- rerun the whole notebook top to bottom.\n\n"
        f"Current working directory: {os.getcwd()}\n"
        "(should be a '.../langchain' directory containing pyproject.toml and src/foundry/)"
    )

from foundry.substrate.budget import BudgetGovernor
from foundry.substrate.db import connect
from foundry.substrate.finding_store import Citation, FindingStore, fingerprint
from foundry.substrate.work_queue import WorkQueue

print(f"'foundry' package found at: {spec.origin}")
print("Setup complete. Continue to the Substrate section below.")

## Substrate

The non-agent machinery every role will depend on: the finding store, the work queue, and the budget governor. No LLM calls in this section — the point is to see the constitution's structural guarantees (not prompt instructions) hold on their own, before any agent touches them.

The same assertions live in `tests/test_finding_store.py` if you'd rather run them as a suite (`pytest tests/ -v`) — this section is the same proofs, run interactively so you can see the state change at each step.

In [ ]:
import tempfile
import time
import threading
from pathlib import Path

from foundry.substrate.db import connect
from foundry.substrate.finding_store import Citation, FindingStore, fingerprint
from foundry.substrate.work_queue import WorkQueue
from foundry.substrate.budget import BudgetCaps, BudgetGovernor

db_path = Path(tempfile.mkdtemp()) / "foundry.sqlite3"
print(f"Using scratch database: {db_path}")

## Constitution VIII — Fingerprints Are Stable Under Edit

A finding's identity is `(normalized_path, symbol, vulnerability_class)` — never a line number or snippet. Re-queueing the same candidate after an unrelated edit (simulated here by changing only the description) must not create a duplicate.

In [ ]:
conn = connect(db_path)
store = FindingStore(conn)

id1, fp1, was_new1 = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="get_user_by_name",
    vulnerability_class="sql-injection",
    description="Detected on first sweep",
    technique="codeguard-rule:input-validation-injection",
)
print(f"First queue:  id={id1} fingerprint={fp1} was_new={was_new1}")

# Simulate a re-run after the function moved a few lines -- only the
# description text differs, identity fields are unchanged.
id2, fp2, was_new2 = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="get_user_by_name",
    vulnerability_class="sql-injection",
    description="Detected on second sweep, function now 4 lines lower",
    technique="codeguard-rule:input-validation-injection",
)
print(f"Second queue: id={id2} fingerprint={fp2} was_new={was_new2}")
assert id1 == id2 and not was_new2, "should have deduplicated, not re-filed"

## Constitution I — Evidence Over Assertion

`assign_verdict()` will not accept `true-positive` unless every citation resolves against a resolver. Here the resolver is a small fake symbol table standing in for the real Indexer (built in the Indexer section, appended later in this notebook) — the mechanism is identical either way. First: a clean pass. Then: a deliberately fabricated citation, to watch it get demoted rather than silently accepted.

In [ ]:
known_symbols = {"get_user_by_name", "users_endpoint", "read_uploaded_file", "files_endpoint"}

def fake_resolver(c: Citation) -> bool:
    return c.symbol in known_symbols

clean_citations = [
    Citation("data/toy_target/vulnerable_app.py", "users_endpoint", "reachability"),
    Citation("data/toy_target/vulnerable_app.py", "get_user_by_name", "impact"),
]
verdict = store.assign_verdict(id1, "true-positive", clean_citations, "clean investigation", fake_resolver)
print(f"Clean citations -> verdict: {verdict}")
assert verdict == "true-positive"

In [ ]:
id3, _, _ = store.queue_candidate(
    normalized_path="data/toy_target/vulnerable_app.py",
    symbol="read_uploaded_file",
    vulnerability_class="path-traversal",
    description="candidate",
    technique="exploratory",
)

fabricated_citations = [
    Citation("data/toy_target/vulnerable_app.py", "sanitize_path_properly", "reachability"),  # does not exist
    Citation("data/toy_target/vulnerable_app.py", "read_uploaded_file", "impact"),
]
verdict = store.assign_verdict(id3, "true-positive", fabricated_citations, "confident but wrong", fake_resolver)
print(f"Fabricated citation -> verdict: {verdict}")
assert verdict == "needs-review", "should have been demoted, not accepted as true-positive"

row = store.get(id3)
print("\nRecorded investigation report:\n", row["investigation_report"])

## Constitution IV — Claims Are Atomic And Mortal

Enqueue several tasks, then race multiple worker threads (each with its own SQLite connection, simulating separate agent processes) to claim them. No task should ever be claimed by more than one worker, and none should be lost.

In [ ]:
queue = WorkQueue(conn, lease_seconds=60)
n_tasks, n_workers = 25, 8
task_ids = {queue.enqueue("index_function", {"i": i}) for i in range(n_tasks)}

claimed_by: dict[int, list[str]] = {}
lock = threading.Lock()

def worker(worker_id: str) -> None:
    wconn = connect(db_path)
    wqueue = WorkQueue(wconn, lease_seconds=60)
    while True:
        task = wqueue.claim_next(worker_id, task_type="index_function")
        if task is None:
            break
        with lock:
            claimed_by.setdefault(task.id, []).append(worker_id)
        wqueue.release(task.id, worker_id, status="done")
    wconn.close()

threads = [threading.Thread(target=worker, args=(f"worker-{i}",)) for i in range(n_workers)]
[t.start() for t in threads]
[t.join(timeout=30) for t in threads]

assert set(claimed_by.keys()) == task_ids, "every task should be claimed exactly once, none lost"
assert all(len(w) == 1 for w in claimed_by.values()), "no task should ever be double-claimed"
print(f"{len(task_ids)} tasks, {n_workers} racing workers -> every task claimed exactly once. No duplicates, none lost.")

## Constitution III — Liveness By Heartbeat, Never By Clock

A claim is only reclaimable once its lease has expired -- never on a fixed wall-clock timeout. Below: a task claimed under a zero-second lease becomes immediately reclaimable; a task claimed under a normal lease does not.

In [ ]:
stale_queue = WorkQueue(conn, lease_seconds=0)  # lease expires immediately
stale_task_id = stale_queue.enqueue("probe", {})
claimed = stale_queue.claim_next("agent-a", task_type="probe")
time.sleep(1.1)

reclaimer_conn = connect(db_path)
reclaimer_queue = WorkQueue(reclaimer_conn, lease_seconds=60)
reclaimed = reclaimer_queue.claim_next("agent-b", task_type="probe")
print(f"Stale claim reclaimed by agent-b: {reclaimed is not None and reclaimed.id == stale_task_id}")
assert reclaimed is not None and reclaimed.id == stale_task_id

fresh_queue = WorkQueue(conn, lease_seconds=60)
fresh_task_id = fresh_queue.enqueue("probe", {})
fresh_queue.claim_next("agent-c", task_type="probe")
stolen = reclaimer_queue.claim_next("agent-d", task_type="probe")
print(f"Fresh claim stolen while agent-c still heartbeating: {stolen is not None and stolen.id == fresh_task_id}")
assert not (stolen is not None and stolen.id == fresh_task_id), "a live claim should not be reclaimable"

## Constitution VI — Coverage Before Yield

`should_stop()` is a conjunction: low yield alone never halts the fleet while coverage is incomplete. Three scenarios: incomplete coverage with zero yield (must not stop), complete coverage with low yield (must stop), complete coverage with healthy yield (must not stop).

In [ ]:
gov = BudgetGovernor(conn, BudgetCaps(yield_threshold=0.5))
gov.record_spend(100.0, "detector sweep so far")

stop, reason = gov.should_stop(coverage_complete=False)
print(f"Coverage incomplete, zero yield -> stop={stop} ({reason})")
assert stop is False, "must not stop on yield alone while coverage is incomplete"

stop, reason = gov.should_stop(coverage_complete=True)
print(f"Coverage complete, zero yield -> stop={stop} ({reason})")
assert stop is True

## Substrate section: recap

Every principle above held under a live workload, with concurrent threads standing in for a real multi-agent fleet — no LLM was involved anywhere in this section. The **Indexer** section comes next, appended below in this same notebook: the first real OpenAI-backed agent, reading `data/toy_target/vulnerable_app.py` and exposing it to the rest of the fleet through the query interface spec.md §5.2 requires.